#### Data set trends and insights

Dataset: 
 
 - _music_featured.csv_
 - _music_location.csv_
 - _music_location_datetime_.csv

Author: Luis Sergio Pastrana Lemus  
Date: 2025-04-23

# Trends and Insights – Music Activity Dataset

## __1. Libraries__

In [1]:
from IPython.display import display, HTML
import os
import pandas as pd
from pathlib import Path
import plotly.express as px
import sys

# Define project root dynamically, gets the current directory from whick the notebook belongs and moves one level upper
project_root = Path.cwd().parent

# Add src to sys.path if it is not already
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Import function directly (more controlled than import *)
from src import *

## __2. Path to Data file__

In [2]:
# Build route to data file and upload
data_file_path = project_root / "data" / "processed"
df_music_featured = load_dataset_from_csv(data_file_path, "music_featured.csv", sep=',', header='infer', keep_default_na=False)
df_music_loc = load_dataset_from_csv(data_file_path, "music_location.csv", sep=',', header='infer', keep_default_na=False)
df_music_loc_dt = load_dataset_from_csv(data_file_path, "music_location_datetime.csv", sep=',', header='infer', keep_default_na=False)

## __3. Casting Data Types__

### 3.1 Casting to category datatype

In [3]:
# Casting to category dtype 
df_music_featured = cast_datatypes(df_music_featured, 'category', c_include=['genre'])
df_music_featured = cast_datatypes(df_music_featured, 'category', c_include=['city'])
df_music_featured = cast_datatypes(df_music_featured, 'category', c_include=['day'])


### 3.2 Casting to datetime data type

In [4]:
# Casting to datetime dtype 
df_music_featured = cast_datatypes(df_music_featured, 'datetime', date_format="%H:%M:%S", c_include=['time'])


In [5]:
df_music_featured

,userid,track,artist,genre,city,time,day,hour
0,FFB692EC,kamigata_to_boots,the_mass_missile,rock,shelbyville,20:28:33,wednesday,20
1,55204538,delayed_because_of_accident,andreas_ronnberg,rock,springfield,14:07:09,friday,14
2,20EC38,funiculi_funicula,mario_lanza,pop,shelbyville,20:58:07,wednesday,20
3,A3DD03C9,dragons_in_the_sunset,fire_ice,folk,shelbyville,08:37:09,monday,8
4,E2DC1FAE,soul_people,space_echo,dance,springfield,08:34:34,monday,8
...,...,...,...,...,...,...,...,...
59957,729CBB09,my_name,mclean,rnb,springfield,13:32:28,wednesday,13
59958,D08D4A55,maybe_one_day_feat_black_spade,blu_exile,hiphop,shelbyville,10:00:00,monday,10
59959,C5E3A0D5,jalopiina,unknown,industrial,springfield,20:09:26,friday,20
59960,321D0506,freight_train,chas_mcdevitt,rock,springfield,21:43:59,friday,21


In [6]:
df_music_featured.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59962 entries, 0 to 59961
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   userid  59962 non-null  object  
 1   track   59962 non-null  object  
 2   artist  59962 non-null  object  
 3   genre   59962 non-null  category
 4   city    59962 non-null  category
 5   time    59962 non-null  object  
 6   day     59962 non-null  category
 7   hour    59962 non-null  int64   
dtypes: category(3), int64(1), object(4)
memory usage: 2.5+ MB


In [7]:
type(df_music_featured['time'].iloc[0])

datetime.time

## __4. Trends and Insights Analysis__

### 4.1 Music Activity: City - Users Location Trends

In [8]:
# Get User's location distribution
music_city_users = df_music_featured.groupby('city', observed=True)['userid'].nunique().reset_index()
music_city_users = music_city_users.rename(columns={'userid': 'users'})
music_city_users

,city,users
0,shelbyville,12289
1,springfield,29061


In [10]:
plot_vertical_bar_plotpx(music_city_users, x='city', y='users', title="Users' Distribution by City", xlabel='City', ylabel='Users Amount', sort=False)

### 4.2 Music Activity: City - Genre Trends

In [11]:
# Get global genre distribution 
df_music_genre = df_music_featured.groupby('genre', observed=True)['userid'].nunique().reset_index()
df_music_genre = df_music_genre.rename(columns={'userid': 'users'}).sort_values(by='users', ascending=False)
df_music_genre

,genre,users
179,pop,7180
53,dance,5161
206,rock,4992
72,electronic,4754
113,hiphop,2606
...,...,...
230,specialty,1
240,taraftar,1
239,tanzorchester,1
237,synthpop,1


In [21]:
plot_vertical_bar_plotpx(df_music_genre.head(50), x='genre', y='users', title='Global Music Genre Distribution (Top 50)', xlabel='Genre', ylabel='Users Amount', sort=False)

In [13]:
df_music_city_genre = df_music_featured.groupby(['city', 'genre'], observed=True)['userid'].nunique().reset_index()
df_music_city_genre = df_music_city_genre.rename(columns={'userid': 'users'})
df_music_city_genre

,city,genre,users
0,shelbyville,acoustic,2
1,shelbyville,adult,8
2,shelbyville,africa,4
3,shelbyville,alternative,591
4,shelbyville,ambient,59
...,...,...,...
444,springfield,videogame,62
445,springfield,vocal,56
446,springfield,western,62
447,springfield,world,1190


In [14]:
# Get Shelbyville genre distribution
df_shelbyville_city_genre = df_music_city_genre.loc[(df_music_city_genre['city'] == 'shelbyville'), :].sort_values(by='users', ascending=False)
df_shelbyville_city_genre


,city,genre,users
136,shelbyville,pop,2090
157,shelbyville,rock,1595
40,shelbyville,dance,1543
55,shelbyville,electronic,1480
88,shelbyville,hiphop,814
...,...,...,...
182,shelbyville,traditional,1
185,shelbyville,tribal,1
186,shelbyville,trip,1
194,shelbyville,vi,1


In [22]:
plot_vertical_bar_plotpx(df_shelbyville_city_genre.head(50), x='genre', y='users', title='Shelbyville Music Genre Distribution (Top 50)', xlabel='Genre', ylabel='Users Amount', sort=False)

In [16]:
# Get Springfield genre distribution
df_springfield_city_genre = df_music_city_genre.loc[(df_music_city_genre['city'] == 'springfield'), :].sort_values(by='users', ascending=False)
df_springfield_city_genre

,city,genre,users
370,springfield,pop,5090
252,springfield,dance,3618
394,springfield,rock,3398
271,springfield,electronic,3276
308,springfield,hiphop,1793
...,...,...,...
423,springfield,synthpop,1
430,springfield,top,1
425,springfield,tanzorchester,1
426,springfield,taraftar,1


In [23]:
plot_vertical_bar_plotpx(df_springfield_city_genre.head(50), x='genre', y='users', title='Springfield Music Genre Distribution (Top 50)', xlabel='Genre', ylabel='Users Amount', sort=False)

### 4.3 Music Activity: City - Track Trends

In [18]:
# Get global track distribution
df_music_track = df_music_featured.groupby('track', observed=True)['userid'].nunique().reset_index()
df_music_track = df_music_track.rename(columns={'userid': 'users'}).sort_values(by='users', ascending=False)
df_music_track

,track,users
4594,brand,126
29735,so_long,109
12626,going_back,89
21818,moscow_calling,87
1423,all_for_you,80
...,...,...
38764,zuni,1
38763,zungguzungguguzungguzeng,1
38762,zumbar,1
5,01_sport_beat_intro_and_instructions_v02,1


In [24]:
plot_vertical_bar_plotpx(df_music_track.head(50), x='track', y='users', title='Global Music Track Distribution (Top 50)', xlabel='Track', ylabel='Users Amount', sort=False)

In [25]:
df_music_city_track = df_music_featured.groupby(['city', 'track'], observed=True)['userid'].nunique().reset_index()
df_music_city_track = df_music_city_track.rename(columns={'userid': 'users'})
df_music_city_track

,city,track,users
0,shelbyville,00,1
1,shelbyville,01,1
2,shelbyville,01_sport_beat_intro_and_instructions_v02,1
3,shelbyville,06_rival,1
4,shelbyville,0_finance,1
...,...,...,...
43401,springfield,zuviel_gluck,1
43402,springfield,zveno_dub,1
43403,springfield,zvezdopad,2
43404,springfield,zvikuru_kuru,1


In [26]:
# Get Shellbyville track distribution
df_shelbyville_city_track = df_music_city_track.loc[(df_music_city_track['city'] == 'shelbyville'), :].sort_values(by='users', ascending=False)
df_shelbyville_city_track

,city,track,users
1702,shelbyville,brand,38
494,shelbyville,all_for_you,28
4718,shelbyville,going_back,27
11086,shelbyville,so_long,27
8130,shelbyville,moscow_calling,24
...,...,...,...
14328,shelbyville,zeze,1
14329,shelbyville,zhanym,1
14330,shelbyville,zhong_xia_ye_zhi_meng,1
14331,shelbyville,zillertaler,1


In [27]:
plot_vertical_bar_plotpx(df_shelbyville_city_track.head(50), x='track', y='users', title='Shelbyville Music Track Distribution (Top 50)', xlabel='Track', ylabel='Users Amount', sort=False)

In [28]:
# Get Springfiel track distribution
df_springfield_city_track = df_music_city_track.loc[(df_music_city_track['city'] == 'springfield'), :].sort_values(by='users', ascending=False)
df_springfield_city_track

,city,track,users
17777,springfield,brand,88
36600,springfield,so_long,82
30656,springfield,moscow_calling,63
23782,springfield,going_back,62
16609,springfield,balenciaga,56
...,...,...,...
43399,springfield,zusammen_feat_clueso_extended,1
43400,springfield,zutter_gd_t_o_p,1
43401,springfield,zuviel_gluck,1
43402,springfield,zveno_dub,1


In [29]:
plot_vertical_bar_plotpx(df_springfield_city_track.head(50), x='track', y='users', title='Springfield Music Track Distribution (Top 50)', xlabel='Track', ylabel='Users Amount', sort=False)

### 4.4 Music Activity: City - Date Trends

In [30]:
# Get global date distribution
df_music_date = df_music_featured.groupby('day', observed=True)['userid'].nunique().reset_index()
df_music_date = df_music_date.rename(columns={'userid': 'users'}).sort_values(by='users', ascending=False)
df_music_date

,day,users
0,friday,16468
1,monday,15921
2,wednesday,13485


In [31]:
plot_vertical_bar_plotpx(df_music_date, x='day', y='users', title='Global Music Date Distribution', xlabel='Day', ylabel='Users Amount', sort=False)

In [32]:
df_music_city_datetime = df_music_featured.groupby(['city', 'day', 'hour'], observed=True)['userid'].nunique().reset_index()
df_music_city_datetime = df_music_city_datetime.rename(columns={'userid': 'users'})
df_music_city_datetime

,city,day,hour,users
0,shelbyville,friday,8,689
1,shelbyville,friday,9,730
2,shelbyville,friday,10,18
3,shelbyville,friday,13,853
4,shelbyville,friday,14,871
5,shelbyville,friday,15,16
6,shelbyville,friday,20,839
7,shelbyville,friday,21,806
8,shelbyville,friday,22,17
9,shelbyville,monday,8,644


In [38]:
# Get shelbyville date distribution
df_shelbyville_city_date = df_music_city_datetime.loc[(df_music_city_datetime['city'] == 'shelbyville'), :]
df_shelbyville_city_date = df_shelbyville_city_date.sort_values(by='users', ascending=False)
df_shelbyville_city_date

,city,day,hour,users
22,shelbyville,wednesday,14,1074
24,shelbyville,wednesday,20,1007
21,shelbyville,wednesday,13,985
25,shelbyville,wednesday,21,919
4,shelbyville,friday,14,871
13,shelbyville,monday,14,867
3,shelbyville,friday,13,853
6,shelbyville,friday,20,839
19,shelbyville,wednesday,9,836
15,shelbyville,monday,20,827


In [39]:
plot_vertical_bar_plotpx(df_shelbyville_city_date, x='day', y='users', title='Shelbyville Music Date Distribution', xlabel='Day', ylabel='Users Amount', sort=False)

In [40]:
# Get Springfield date distribution
df_springfield_city_date = df_music_city_datetime.loc[(df_music_city_datetime['city'] == 'springfield'), :]
df_springfield_city_date = df_springfield_city_date.sort_values(by='users', ascending=False)
df_springfield_city_date

,city,day,hour,users
28,springfield,friday,9,2216
31,springfield,friday,14,2205
33,springfield,friday,20,2201
27,springfield,friday,8,2196
42,springfield,monday,20,2188
36,springfield,monday,8,2165
40,springfield,monday,14,2137
37,springfield,monday,9,2123
30,springfield,friday,13,2089
39,springfield,monday,13,2075


In [41]:
plot_vertical_bar_plotpx(df_springfield_city_date, x='day', y='users', title='Springfield Music Date Distribution', xlabel='Day', ylabel='Users Amount', sort=False)

### 4.5 Music Activity: City - Time Trends

In [42]:
# Get global time distribution
df_music_time = df_music_city_datetime.groupby(['city', 'hour'], observed=True)['users'].sum().reset_index()
df_music_time = df_music_time.sort_values(by='users', ascending=False)
df_music_time



,city,hour,users
13,springfield,14,6057
15,springfield,20,6030
12,springfield,13,5826
10,springfield,9,5611
9,springfield,8,5567
16,springfield,21,5311
4,shelbyville,14,2812
6,shelbyville,20,2673
3,shelbyville,13,2607
7,shelbyville,21,2493


In [43]:
plot_vertical_bar_plotpx(df_music_time, x='hour', y='users', title='Global Music Time Distribution', xlabel='Hour', ylabel='Users Amount', sort=False)

In [44]:
# Get shelbyville time distribution
df_shelbyville_city_time = df_music_time.loc[(df_music_time['city'] == 'shelbyville'), :]
df_shelbyville_city_time = df_shelbyville_city_time.sort_values(by='users', ascending=False)
df_shelbyville_city_time

,city,hour,users
4,shelbyville,14,2812
6,shelbyville,20,2673
3,shelbyville,13,2607
7,shelbyville,21,2493
1,shelbyville,9,2221
0,shelbyville,8,2148
8,shelbyville,22,56
5,shelbyville,15,47
2,shelbyville,10,41


In [45]:
plot_vertical_bar_plotpx(df_shelbyville_city_date, x='hour', y='users', title='Shelbyville Music Time Distribution', xlabel='Hour', ylabel='Users Amount', sort=False)

In [46]:
# Get springfield time distribution
df_springfield_city_time = df_music_time.loc[(df_music_time['city'] == 'springfield'), :]
df_springfield_city_time = df_springfield_city_time.sort_values(by='users', ascending=False)
df_springfield_city_time

,city,hour,users
13,springfield,14,6057
15,springfield,20,6030
12,springfield,13,5826
10,springfield,9,5611
9,springfield,8,5567
16,springfield,21,5311
14,springfield,15,120
11,springfield,10,120
17,springfield,22,104


In [47]:
# Get sprinfield time distribution
plot_vertical_bar_plotpx(df_springfield_city_time, x='hour', y='users', title='Springfield Music Time Distribution', xlabel='hour', ylabel='Users Amount', sort=False)

### 4.6 Music Activity: City - Date-Time Trends

In [49]:
# Get shelbyville date-time distribution
df_shelbyville_city_time = df_music_city_datetime.loc[(df_music_city_datetime['city'] == 'shelbyville'), :]
df_shelbyville_city_time['date_time'] = df_shelbyville_city_time['day'].astype(str) + '_' + df_shelbyville_city_time['hour'].astype(str)
df_shelbyville_city_time

C:\Users\luisp\AppData\Local\Temp\ipykernel_14876\3917481107.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,city,day,hour,users,date_time
0,shelbyville,friday,8,689,friday_8
1,shelbyville,friday,9,730,friday_9
2,shelbyville,friday,10,18,friday_10
3,shelbyville,friday,13,853,friday_13
4,shelbyville,friday,14,871,friday_14
5,shelbyville,friday,15,16,friday_15
6,shelbyville,friday,20,839,friday_20
7,shelbyville,friday,21,806,friday_21
8,shelbyville,friday,22,17,friday_22
9,shelbyville,monday,8,644,monday_8


In [50]:
plot_vertical_bar_plotpx(df_shelbyville_city_time, x='date_time', y='users', title='Shelbyville Music Date And Time distribution', xlabel='Date And Time', ylabel='Users Amount', sort=False)

In [51]:
# Get shellbyville date-time distribution
df_springfield_city_time = df_music_city_datetime.loc[(df_music_city_datetime['city'] == 'springfield'), :]
df_springfield_city_time['date_time'] = df_springfield_city_time['day'].astype(str) + '_' + df_springfield_city_time['hour'].astype(str)
df_springfield_city_time

C:\Users\luisp\AppData\Local\Temp\ipykernel_14876\665467705.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,city,day,hour,users,date_time
27,springfield,friday,8,2196,friday_8
28,springfield,friday,9,2216,friday_9
29,springfield,friday,10,52,friday_10
30,springfield,friday,13,2089,friday_13
31,springfield,friday,14,2205,friday_14
32,springfield,friday,15,44,friday_15
33,springfield,friday,20,2201,friday_20
34,springfield,friday,21,1989,friday_21
35,springfield,friday,22,40,friday_22
36,springfield,monday,8,2165,monday_8


In [52]:
plot_vertical_bar_plotpx(df_springfield_city_time, x='date_time', y='users', title='Springfield Music Date And Time distribution', xlabel='Date And Time', ylabel='Users Amount', sort=False)

## 5. Conclusions and key insights

This exploratory data analysis reveals several important insights into music activity across cities, days, and time periods:

### 🧠 Conclusion: User Activity Patterns by Day and City

Analysis of user engagement across different cities and weekdays reveals clear behavioral differences. 

- Springfield shows higher music activity compared to Shelbyville.
- When comparing overall genre preferences, both Springfield and Shelbyville share similar tastes — the most listened genres are Pop, Dance, Rock, and Electronic.
- In terms of tracks, both cities show similar listening patterns, with “Brand,” “So_Long,” and “Going_Back” being the most played songs.
- Springfield users listen to more music on Fridays, while Shelbyville users peak on Wednesdays.
- Both Springfield and Shelbyville have the highest user activity around 14:00.
- Springfield shows a more stable listening trend on Fridays and Mondays, whereas Shelbyville displays greater variability, with Wednesday being its most active day.